# Module 4.2: The Decoder Layer

In the previous module, we built the **Encoder**, which is essentially a "Reader". It reads an entire sentence at once in massive parallel matrix operations to build a deep understanding of the context.

Now we build the **Decoder**, the "Writer". The Decoder takes the context from the Encoder and writes out a response (like translating French to English, or ChatGPT generating an answer). But writing is fundamentally different from reading: you can only write one word at a time, and you cannot look into the future.

> ⚠️ **Reminder (see Module 4.1):** we are still building the *historical* 2017 Encoder–Decoder Transformer. The **Cross-Attention** layer you are about to learn bridges the Encoder and Decoder — but from Module 4.3 onward we go *decoder-only* (Llama-style) and **delete Cross-Attention entirely**. Learn it for understanding, not because the final model keeps it.

## 1. The Generation Problem (Autoregressive)

### The Concept
A Decoder is "autoregressive". This means it predicts word 1, feeds word 1 back into itself to predict word 2, feeds words 1 & 2 back to predict word 3, and so on. 

### Why do we need a new layer? (The Cheating Problem)
During training, we give neural networks the *entire* correct answer paragraph right away for insane mathematical speed. But if we let a Decoder use standard Attention (where every word looks at every other word), the first word being generated would simultaneously "look" at the words coming after it in the training data! 
The model would "cheat" by looking into the future, guaranteeing it gets the answers right in training, but utterly failing when you actually try to talk to it in the real world.

## 2. Causal Masked Attention (The Blinder)

### The Analogy (The Test Paper)
Imagine taking a test where the answers are printed on the right side of the page. To test yourself honestly, you slide a piece of cardboard down line-by-line, covering everything below where you are reading. You can see the past, but the future is "blacked out."

### The Math
Before we run the Softmax function, we force the Attention Scores of all "future" words to a very large negative number. In code we use `-1e9` (a numerically-stable stand-in for `-inf`). Mathematically, `softmax` turns such a large negative score into ~`0.0`, so future words contribute essentially `0.0%` to the context sum.
We build the "allowed" positions using a lower-triangular matrix, known in PyTorch as `torch.tril`.

### Why do we need it?
Mathematically enforcing a sequence direction allows us to feed thousands of words into the matrix simultaneously during training, fully unlocking the GPU's parallel processing speed, while enforcing the strict rule that words can only ever look backwards, never forwards.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)  # Reproducibility

# Imagine a sentence with 5 words. We want to build a mask for it.
seq_len = 5

# Create a mask of 1s (True) in the lower triangle, and 0s (False) in the upper triangle
subsequent_mask = torch.tril(torch.ones(seq_len, seq_len))

print("1.0 means 'I can look here'. 0.0 means 'MASKED!'")
print(subsequent_mask)

# Visualizing the Mask.
# We use a reversed colormap so that DARK = blocked (matches the "cardboard over
# the future answers" analogy) and light = visible.
plt.figure(figsize=(5,5))
plt.imshow(subsequent_mask.numpy(), cmap='Greys_r')  # 0 -> dark (blocked), 1 -> light (allowed)
plt.title('Causal Attention Mask (dark = blocked future)')
plt.xlabel('Keys (Words to look at)')
plt.ylabel('Queries (Current Word)')
plt.xticks(range(5), ['Word 1', 'Word 2', 'Word 3', 'Word 4', 'Word 5'], rotation=45)
plt.yticks(range(5), ['Word 1', 'Word 2', 'Word 3', 'Word 4', 'Word 5'])
plt.colorbar(ticks=[0, 1], label='0 = blocked, 1 = allowed')
plt.show()

### Watching the mask actually work

The mask is only useful if it really zeroes out the future. Let's *call* our `scaled_dot_product_attention` with the 5×5 causal mask and print the resulting attention weights. The upper-right triangle should be exactly `0.0` — proof that no word can see the future.

In [ ]:
import torch.nn.functional as F
from math import sqrt

def scaled_dot_product_attention(query, key, value, mask=None):
    dk = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / sqrt(dk)
    if mask is not None:
        # Wherever the mask is 0 (a future position), overwrite the score with -1e9.
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights

torch.manual_seed(0)
d_k = 8
q = torch.randn(seq_len, d_k)
k = torch.randn(seq_len, d_k)
v = torch.randn(seq_len, d_k)

_, weights = scaled_dot_product_attention(q, k, v, mask=subsequent_mask)

print("Attention weights (rows = current word, cols = word being looked at):")
print(weights.round(decimals=2))
print("\nNotice the entire upper-right triangle is 0.00 -> the future is invisible.")
print("Each row sums to 1.0:", weights.sum(dim=-1).round(decimals=2).tolist())

## 3. Cross-Attention (The Translator)

### The Concept
Now our Decoder is blindly generating text strictly based on the previous words it just wrote. But wait... how does it know what it's supposed to be writing about? If it's translating French (Encoder) to English (Decoder), how do they communicate?

### The Analogy (The Detective)
Imagine a Detective (the Decoder) trying to write a case report. They know what they are writing right now (the **Query**), but they need to pull facts from the massive library of evidence gathered earlier by the Police (the Encoder's final **Keys** and **Values**).

### Why do we need it?
Cross-attention bridges the two networks. 
- The `Query (Q)` matrix is generated directly from the Decoder's own self-attention output.
- The `Key (K)` and `Value (V)` matrices are piped in directly from the final output of the Encoder.
This allows the English word that is currently being typed to dynamically "sweep" over the entire French sentence to find the relevant context before it prints!

### ⚠️ A shape subtlety: output length follows the QUERIES
In cross-attention the queries and the keys/values come from *different* sequences and can have *different lengths*. Suppose the Decoder has written **3** words (3 queries) and the Encoder produced **10** French vectors (10 keys/values):
- The attention-weight matrix has shape `(3 queries × 10 keys)` — each query attends over all 10 keys.
- But the **output has one vector per QUERY**, so its length is **3**, *not* 10.

In general: **cross-attention output length = number of queries (decoder length)**, regardless of how many keys/values the encoder provides. Watch for this in the shapes printed below.

## 4. Building the Full Decoder Block

### The Concept
The Decoder is just an Encoder Block with one extra layer squeezed into the middle.

1. **Masked Self-Attention**: It looks at what it has written so far.
2. **Cross-Attention**: It queries the Encoder for relevant translation details.
3. **Feed-Forward**: It processes this combined information.

> **A note on normalization:** Module 4.1 built and used our own `RMSNorm`. Here we use PyTorch's built-in `nn.LayerNorm` purely to keep this cell short and dependency-free. They play the same role (a normalization step); a production decoder-only model would reuse the `RMSNorm` from Module 4.1.

In [ ]:
import torch.nn as nn

# Note: In a full pipeline, we would import RMSNorm and FeedForward from the Encoder Layer (Notebook 09),
# and a real MultiHeadAttention from Notebook 08. For a short, self-contained demo we mock
# attention with a single Linear layer (right shapes, but no real Q/K/V math).

class MockMultiHeadAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj_q = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        # A real implementation would compute scaled_dot_product_attention(q, k, v, mask)
        # across multiple heads. This mock just projects the queries so the output length
        # follows the QUERIES (decoder length) -- the key shape behavior to remember.
        return self.proj_q(q)

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, hidden_dim: int):
        super().__init__()

        # 1. Masked Self-Attention (Look only at the past)
        self.masked_self_attention = MockMultiHeadAttention(d_model)
        self.norm1 = nn.LayerNorm(d_model)  # built-in LayerNorm to keep the cell short

        # 2. Cross-Attention (Look at the Encoder)
        self.cross_attention = MockMultiHeadAttention(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # 3. Memory Processor (Feed-Forward)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model),
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, causal_mask):
        # Step 1: Pre-Norm Masked Self-Attention (Q, K, V all come from 'x')
        norm_x = self.norm1(x)
        x = x + self.masked_self_attention(norm_x, norm_x, norm_x, mask=causal_mask)

        # Step 2: Cross-Attention (Q comes from the Decoder; K and V come from the Encoder)
        norm_x2 = self.norm2(x)
        x = x + self.cross_attention(norm_x2, encoder_output, encoder_output)

        # Step 3: Feed-Forward (the "thinker")
        x = x + self.ffn(self.norm3(x))

        return x

# --- Let's Test It Out! ---
d_model = 128

# Pretend the Encoder processed a 10-word French sentence.
encoder_french_output = torch.randn(1, 10, d_model)
# Pretend the Decoder has generated 3 English words so far.
decoder_english_input = torch.randn(1, 3, d_model)

# Mask to stop the Decoder from cheating (3x3 for the 3 generated words)
causal_mask = torch.tril(torch.ones(3, 3))

# Instantiate the Block
decoder = DecoderBlock(d_model=d_model, num_heads=8, hidden_dim=512)

# Forward Pass!
output = decoder(decoder_english_input, encoder_french_output, causal_mask)
print(f"Decoder Input length: {decoder_english_input.shape}  (3 queries)")
print(f"Encoder Input length: {encoder_french_output.shape}  (10 keys/values)")
print(f"Final Output length:  {output.shape} -> (Batch, Seq_Len_English, Embed_Dim)")
print("\nNote: output length is 3 (the DECODER length), not 10, even though we attended over 10 encoder keys.")

### 🏋️ Try it yourself

**Task 1 — Prove the length rule.** Keep the decoder input at 3 words but change the encoder output to 20 words (`torch.randn(1, 20, d_model)`). Run the block. Does the final output length change? Explain why in one sentence.

**Task 2 — Break the mask.** Re-run the `scaled_dot_product_attention` demo with `mask=None`. Print the attention weights and confirm the upper-right triangle is no longer 0 — i.e. the model is now "cheating" by looking into the future.

In [ ]:
# Task 1 starter: longer encoder, same decoder.
torch.manual_seed(0)
longer_encoder_output = torch.randn(1, 20, d_model)   # 20 French words now
decoder_input = torch.randn(1, 3, d_model)            # still 3 English words
mask3 = torch.tril(torch.ones(3, 3))
out = decoder(decoder_input, longer_encoder_output, mask3)
print("Output shape with 20 encoder words:", out.shape, "(still 3 -> follows the queries)")

# Task 2 starter: same attention, but no mask. Compare to the masked version above.
# _, no_mask_weights = scaled_dot_product_attention(q, k, v, mask=None)
# print(no_mask_weights.round(decimals=2))